# Pipeline
This notebook creates a CI/CD pipeline that runs the full MLOps system. Due to some limitations with resource limitability in AWS as well as permissions, this pipeline will take a pretrained model in instead of training as part of the pipeline. Since the problem is a computer vision problem, the training takes much longer and uses too many AWS resources. In order to adjust for this, the model is trained within a CI/CD action in Github. When changes are made to the model, or a new model is tested, the pipeline will run and create an h5 artifact and notify the maintainers that a new artifact exists. This h5 file is uploaded to an S3 bucket (outside of this notebook) and then used in the remaining processing steps. Stages in the pipeline are:
1. Determine Metrics

In [41]:
# Setup steps
import sys

import boto3
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")
sm_client = boto3.client('sagemaker')
pipeline_session = PipelineSession()
model_package_group_name = f"FERModelGroupName"

### Define Parameters for Pipeline Execution
Define the Pipeline parameters that are used for this pipeline. This enables custom pipeline executions without having to modify the pipeline definition. The parameters defined in this workflow are:
- `processing_instance_count` - The instance count of the processing job
- `instance_type` - The `ml.*` instance type of the training job
- `model_approval_status` - The approval status to register with the trained model for CI/CD purposes (Defaults to "PendingManualApproval")
- `accuracy_threshold` - The accuracy threshold used to verify the accuracy of the model, initially set to >50%
- `s3_bucket` - The name of the s3 bucket that holds the data
- `s3_test_data_prefix` - Prefix of the directory that holds the test data (used in evaluation)
- `s3_model_prefix` - Where to find the model in s3

In [42]:
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)

processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")
model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval"
)
accuracy_threshold = ParameterFloat(name="AccuracyThreshold", default_value=0.5)
s3_bucket = ParameterString(name="S3Bucket", default_value="sagemaker-us-east-1-399018723364")
s3_test_data_prefix = ParameterString(name="S3TestDataPrefix", default_value="test/test/")
s3_model_prefix = ParameterString(name="S3ModelPrefix", default_value="group-5/models/")


### Define Model Evaluation Step 
This step evaluates the pre-trained model. This is a custom evaluation script that will preform the model evaluation. After the pipeline runs, the resulting `evaluation.json` file will be available for further analysis. 

### Register Model
A model package is an abstraction of reusable model artifacts that packages all the ingredients required for inference. Primarily, it consistes of an inference specification that defines the inference image to use along with an option model weights location.

A model package group is a collection of model packages. A model package group can be created for a specific ML business problem, and new versions of the model packages can be added to it. Typically, customers are expected to create a ModelPackageGroup for a SageMaker pipeline so that model package versions can be added to the group for every SageMaker pipeline run.

In [43]:
from sagemaker.model import Model

model_data = f's3://{bucket}/group-5/models/model.tar.gz'

image_uri = sagemaker.image_uris.retrieve(
        framework="pytorch",
        region=region,
        version="1.10.0",
        image_scope="inference",
        instance_type="ml.m5.xlarge"
    )

model = Model(
    image_uri=image_uri,
    model_data=model_data,
    sagemaker_session=pipeline_session,
    role=role,
)

INFO:sagemaker.image_uris:Defaulting to only available Python version: py38


In [45]:
import json
# Get Model Metrics from S3
s3_model_metrics = f's3://{bucket}/group-5/metrics/metrics.json'

# s3_client = boto3.client("s3")
# s3_client.download_file(s3_model_metrics)

# with open("metrics.json", "r") as f:
#     metrics_data = json.load(f)



In [46]:
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.parameters import ParameterString

# accuracy = metrics_data.get("Accuracy", 0.0)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=s3_model_metrics,
        content_type="application/json"
    )
)

model_package_arn_param = ParameterString(name="ModelPackageArn")

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.xlarge"],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics
)
step_register = ModelStep(name="FERRegisterModel", step_args=register_args)


/opt/conda/lib/python3.11/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


## Step to Create Endpoint
This step will create a model endpoint now that the model has been registered. 

In [47]:
from datetime import datetime
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.steps import CreateModelStep, ProcessingStep
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.workflow.parameters import ParameterString

# Create the Model in SageMaker
step_create_model = CreateModelStep(
    name="CreateSageMakerModel",
    model=model,
)


### Define a pipeline of parameters, steps and conditions
In this section, the steps defined above are combined into a Pipeline so it can be executed.

A pipeline requires a `name`, `parameters`, and `steps`. 

In [48]:
from sagemaker.workflow.pipeline import Pipeline


pipeline_name = f"FERPipeline-WithEndpoint"
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        instance_type,
        model_approval_status
    ],
    steps=[step_register, step_create_model]
)

print(pipeline.steps)

[ModelStep(name='FERRegisterModel', steps=[<sagemaker.workflow._utils._RegisterModelStep object at 0x7fefce359790>], depends_on=None), <sagemaker.workflow.steps.CreateModelStep object at 0x7fefd7c55350>]


In [35]:
# Check pipeline definition
import json


definition = json.loads(pipeline.definition())
definition

{'Version': '2020-12-01',
 'Metadata': {},
 'Parameters': [{'Name': 'ProcessingInstanceCount',
   'Type': 'Integer',
   'DefaultValue': 1},
  {'Name': 'TrainingInstanceType',
   'Type': 'String',
   'DefaultValue': 'ml.m5.xlarge'},
  {'Name': 'ModelApprovalStatus',
   'Type': 'String',
   'DefaultValue': 'PendingManualApproval'}],
 'PipelineExperimentConfig': {'ExperimentName': {'Get': 'Execution.PipelineName'},
  'TrialName': {'Get': 'Execution.PipelineExecutionId'}},
 'Steps': [{'Name': 'FERRegisterModel-RegisterModel',
   'Type': 'RegisterModel',
   'Arguments': {'ModelPackageGroupName': 'FERModelGroupName',
    'InferenceSpecification': {'Containers': [{'Image': '763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:1.10.0-cpu-py38',
       'Environment': {},
       'ModelDataUrl': 's3://sagemaker-us-east-1-399018723364/group-5/models/model.tar.gz'}],
     'SupportedContentTypes': ['text/csv'],
     'SupportedResponseMIMETypes': ['text/csv'],
     'SupportedRealtimeInferen

### Submit pipeline to SageMaker and Execute
Submit the pipeline definition to the Pipeline service. The Pipeline service uses the role that is passed in to create all the jobs defined in the steps.

In [49]:
pipeline.upsert(role_arn=role)


{'PipelineArn': 'arn:aws:sagemaker:us-east-1:399018723364:pipeline/FERPipeline-WithEndpoint',
 'ResponseMetadata': {'RequestId': '0ccaf6db-9672-4f05-b48e-22675f5a3b6d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '0ccaf6db-9672-4f05-b48e-22675f5a3b6d',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '92',
   'date': 'Sun, 23 Feb 2025 20:14:57 GMT'},
  'RetryAttempts': 0}}

In [50]:
# Start the pipeline and accept all the default parameters
execution = pipeline.start()

### Pipeline Operations: Examining and waiting for pipeline execution
Describe the pipeline and examing the execution

In [38]:
execution.describe()

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:399018723364:pipeline/FERPipeline-WithEndpoint',
 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:399018723364:pipeline/FERPipeline-WithEndpoint/execution/lw8v9yw16u5k',
 'PipelineExecutionDisplayName': 'execution-1740338237833',
 'PipelineExecutionStatus': 'Succeeded',
 'PipelineExperimentConfig': {'ExperimentName': 'ferpipeline-withendpoint',
  'TrialName': 'lw8v9yw16u5k'},
 'CreationTime': datetime.datetime(2025, 2, 23, 19, 17, 17, 739000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2025, 2, 23, 19, 17, 20, 710000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:399018723364:user-profile/d-coj0j6t5xpww/kvierczhalek',
  'UserProfileName': 'kvierczhalek',
  'DomainId': 'd-coj0j6t5xpww',
  'IamIdentity': {'Arn': 'arn:aws:sts::399018723364:assumed-role/LabRole/SageMaker',
   'PrincipalId': 'AROAVZZ26RASKWFFRC3N6:SageMaker'}},
 'LastModifiedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1

In [39]:
execution.wait()

In [40]:
# Shows which steps have been executed
execution.list_steps()

[{'StepName': 'CreateSageMakerModel',
  'StartTime': datetime.datetime(2025, 2, 23, 19, 17, 19, 5000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2025, 2, 23, 19, 17, 20, 432000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'Model': {'Arn': 'arn:aws:sagemaker:us-east-1:399018723364:model/pipelines-lw8v9yw16u5k-CreateSageMakerModel-qEHP3QSb4A'}},
  'AttemptCount': 1},
 {'StepName': 'FERRegisterModel-RegisterModel',
  'StartTime': datetime.datetime(2025, 2, 23, 19, 17, 19, 5000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2025, 2, 23, 19, 17, 20, 132000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'RegisterModel': {'Arn': 'arn:aws:sagemaker:us-east-1:399018723364:model-package/FERModelGroupName/22'}},
  'AttemptCount': 1}]